<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Help the model with your words

**[Help the model with your words](https://learning.nextia-ai.com/courses/speech/m04/help-the-model-with-your-words/)** · Speech AI: Transcribe, Understand and Speak · Module 4, lesson 4 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure what Larkfield's vocabulary prompt does to Whisper base and small, find the calls where it makes things worse, write the check-digit rule for order numbers, and see which checks catch every wrong order number in the transcripts.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (a few MB). No account, no key, and no model download: this notebook reads recorded transcripts. |
| **You should know** | WER, the normalisation guide and the important words, from [Normalise before you compare](https://learning.nextia-ai.com/courses/speech/m02/normalise-before-you-compare/) and [The errors that matter](https://learning.nextia-ai.com/courses/speech/m02/the-errors-that-matter/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m04/help-the-model-with-your-words/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M04-L04-help-the-model-with-your-words/help-the-model-with-your-words-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 3 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/orders.json": "74f313e9e6c6eac155c655f2c4ed5016ae8aa6076c38ca88edf7c09ed901aeda",
    "data/recordings/asr/whisper-tiny.jsonl": "d6c12c960a2bc99981f866d674830b7501c01f754ae0f11d8bacf1139aa7285b",
    "data/recordings/asr/whisper-base.jsonl": "2af177c7cef6bcb9a24dabdcb4c108a785d7cf8effae475de053390ec5465c90",
    "data/recordings/asr/whisper-base-prompt.jsonl": "cd61a14d41b459cac65243d756f53c3d63e6022fda02dcaee083686e39f2e2d9",
    "data/recordings/asr/whisper-small.jsonl": "953f41bdba51fe75c411960af3757a8c545b29670c87ac9f04b25f824ff3cf6d",
    "data/recordings/asr/whisper-small-prompt.jsonl": "afa9714a7ef28aae5f23fc34b4e7bfe853ab9e82a29ae457de7dcecc4543f298",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The recorded transcripts

The course team ran Whisper on the 60 test calls twice: once as it is, and once with Larkfield's **vocabulary prompt**. The prompt is a short text that Whisper reads before the call, as if it had just heard it: "Larkfield help desk. Grow Hub, Solara lantern, Cedar raised bed, … Order 742158." (in French, "Service client Larkfield. …"). Run the next cell. It loads the transcripts and the fake order system.

In [ ]:
calls = load_calls()
test = [c for c in calls.values() if c["split"] == "test"]
methods = ['whisper-tiny', 'whisper-base', 'whisper-base-prompt', 'whisper-small', 'whisper-small-prompt']
recorded = {}   # recorded[method][call_id] = the phone transcript
for m in methods:
    rows = [json.loads(line) for line in open(f"data/recordings/asr/{m}.jsonl", encoding="utf-8") if line.strip()]
    recorded[m] = {r["call_id"]: r["text"] for r in rows if r["kind"] == "phone"}
orders = {o["order"]: o for o in json.load(open("data/orders.json"))}
print(len(test), "test calls;", len(orders), "orders in the fake order system")

> **Check.** 60 test calls; 160 orders.

## 2. Product names, with and without the prompt

> **Predict.** The prompt lists the 12 product names. Will it help both sizes the same way? Will the WER go down for both?

In [ ]:
def missed(method, kind, language=None):
    """How many important items of one kind (product, order, date, negation) the transcripts miss."""
    rows = [cp.important_errors(c["important"], recorded[method][c["call_id"]], c["language"])
            for c in test if language in (None, c["language"])]
    total = cp.sum_important(rows)[kind]
    return total["missed"], total["items"]

def wer(method, language=None):
    items = [c for c in test if language in (None, c["language"])]
    return round(cp.corpus_scores([(reference(c), recorded[method][c["call_id"]], c["language"]) for c in items])["wer"], 3)

for m in ["whisper-base", "whisper-base-prompt", "whisper-small", "whisper-small-prompt"]:
    print(f"{m:21} products missed {missed(m, 'product')}   WER {wer(m)}   English WER {wer(m, 'en')}")

> **Check.** Product names missed (of 60): base 42 → 17, small 41 → 9. But the WER of base goes **up**, from 0.137 to 0.149, and its English WER from 0.051 to 0.079. The prompt helped small everywhere and hurt base.

## 3. What went wrong for base?

A WER mixes three kinds of error. Count them apart for the English calls.

In [ ]:
def error_kinds(method, language="en"):
    """Substitutions, deletions and insertions, summed over the calls of one language."""
    total = {"sub": 0, "del": 0, "ins": 0}
    for c in test:
        if c["language"] != language:
            continue
        k = cp.counts(cp.normalise(reference(c), "full", language),
                      cp.normalise(recorded[method][c["call_id"]], "full", language))
        for key in total:
            total[key] += k[key]
    return total

for m in ["whisper-base", "whisper-base-prompt"]:
    print(f"{m:21}", error_kinds(m))

> **Check.** Without the prompt: {'sub': 125, 'del': 17, 'ins': 63}. With the prompt: {'sub': 92, 'del': 165, 'ins': 58}. The substitutions went down (the names are right more often), but the **deletions** went up about ten times: base skipped whole stretches of some calls.

Read one of them. In call C-0077, the caller asks about an Aqua Mist sprinkler and gives order number 613828.

In [ ]:
for m in ["whisper-base", "whisper-base-prompt"]:
    print(f"--- {m}:")
    print(recorded[m]["C-0077"])

> **Check.** With the prompt, the transcript jumps from "How can I help you today?" straight to "It's shipped on the 6th of June". The caller's question, the agent's question and the order number are gone. A small model can treat the prompt as text that it has already written, and skip ahead. Always measure a prompt on the size you will use.

## 4. A check digit for order numbers

Larkfield's order numbers have 6 digits, and the last one is a **check digit**: it is computed from the other five with the Luhn rule, as on bank cards. So a number with one wrong digit, or two neighbour digits swapped, fails the check before anyone looks it up.

The rule, from the right: keep the 1st digit, double the 2nd, keep the 3rd, double the 4th, and so on. If a doubled digit is over 9, subtract 9. Add everything up. The number is good if the total ends in 0.

> **Your turn.** Write `luhn_ok(number)`. It must print `True False False` for the three examples.

In [ ]:
def luhn_ok(number):
    """True if the last digit of `number` is the right check digit (the Luhn rule)."""
    total = 0
    for i, ch in enumerate(reversed(number)):
        d = int(ch)
        if i % 2 == 1:      # every second digit from the right is doubled
            d *= 2
            if d > 9:
                d -= 9
        total += d
    return total % 10 == 0

print(luhn_ok("742155"), luhn_ok("742151"), luhn_ok("472155"))

In [ ]:
luhn_results = [luhn_ok(o) for o in sorted(orders)[:20]] + [luhn_ok(x) for x in ["742155", "742151", "472155", "384529", "384592"]]
print(sum(bool(x) for x in luhn_results), "of", len(luhn_results), "pass")

In [ ]:
expect_hash('luhn_results', luhn_results, 'fbd01a6643cbcc5e')

> **Check.** The first 20 orders all pass, and 22 of the 25 numbers pass in all. "384592" is "384529" with the last two digits swapped: the check catches it.

## 5. What catches each wrong order number?

After recognition, the IVR can test a heard number three ways: is it 6 digits long, is its check digit right, and is it in the order system? The next cell lists every test call whose order number Whisper small missed, the digit runs it wrote instead, and the check that stops each one.

In [ ]:
import re

def catch(heard):
    """Which check stops a wrong number: the length, the check digit, the lookup, or none."""
    if len(heard) != 6:
        return "length"
    if not luhn_ok(heard):
        return "check digit"
    if heard not in orders:
        return "lookup"
    return "not caught"

def order_report(method):
    """For each call whose order number the transcript misses: the digit runs it holds, and what catches them."""
    report = []
    for c in test:
        if not c["important"]["order"]:
            continue
        true = cp.normalise(c["important"]["order"][0], "full", c["language"])
        text = cp.normalise(recorded[method][c["call_id"]], "full", c["language"])
        if f" {true} " in f" {text} ":
            continue
        heard = sorted({r for r in re.findall(r"\d+", text) if len(r) >= 3 and r != true})
        report.append((c["call_id"], true, heard, [catch(h) for h in heard]))
    return report

for row in order_report("whisper-small"):
    print(row)

> **Check.** Whisper small missed 8 order numbers. Most are **one digit short** ("26026" for 260026: the caller said "oh" or a double digit). The length check stops them.

## 6. Your turn: does any wrong number get through?

> **Your turn.** For `whisper-tiny`, `whisper-base` and `whisper-small`, count the wrong digit runs that pass **all three** checks ("not caught"). Put the counts in a dictionary `not_caught` (method → number). Use `order_report`.

In [ ]:
not_caught = {}
for m in ["whisper-tiny", "whisper-base", "whisper-small"]:
    not_caught[m] = sum(kinds.count("not caught") for _, _, _, kinds in order_report(m))
print(not_caught)

In [ ]:
expect_hash('not_caught', not_caught, '1f3f1d5027555f5b')

> **Check.** None of the wrong numbers passes all three checks on these calls. The checks do not **fix** a number: they tell the IVR to ask again, or to ask for the keypad. That is still worth a lot: the IVR never acts on a wrong order.

## Next

You measured the vocabulary prompt on two sizes, found where it hurts, and checked order numbers after recognition. The [lesson page](https://learning.nextia-ai.com/courses/speech/m04/help-the-model-with-your-words/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Accents, noise and two languages](https://learning.nextia-ai.com/courses/speech/m04/accents-noise-and-two-languages/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m04/help-the-model-with-your-words/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.